# Sesión 1 — Google Gen AI SDK: fundamentos y entradas multimodales

**Rodrigo López Vera — Data Science, Revolut Perú**  
Curso GenAI Multimodal · Posgrado Data/IA · Mayo 2026

---

### Lo que vas a construir hoy

Al final de esta sesión vas a poder:
1. Llamar a Gemini desde código Python y controlar su comportamiento
2. Extraer información estructurada (JSON) de texto y de imágenes
3. Enviarle PDFs y recibir respuestas sobre su contenido
4. Convertir texto en embeddings y calcular similitud semántica a mano
5. Construir un sistema de compliance básico sin ninguna librería de RAG

**Stack:** `google-genai` · `numpy` · `Pillow`  
**Modelos:** `gemini-3.1-flash-lite-preview` (generación) · `gemini-embedding-2` (embeddings)

---
## 00 · Setup

In [1]:
!pip install google-genai numpy Pillow --quiet


[notice] A new release of pip available: 22.2.2 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
from google import genai
from google.genai import types
from pydantic import BaseModel
from PIL import Image
import numpy as np
import json
import io

def load_api_key() -> str:
    try:
        from dotenv import load_dotenv
        load_dotenv()
        key = os.environ.get("GOOGLE_API_KEY")
        if key:
            print("API key cargada desde .env")
            return key
    except ImportError:
        pass
    try:
        from google.colab import userdata
        key = userdata.get("GOOGLE_API_KEY")
        if key:
            print("API key cargada desde Colab Secrets")
            return key
    except Exception:
        pass
    raise EnvironmentError(
        "No se encontró GOOGLE_API_KEY.\n"
        "Local: archivo .env con GOOGLE_API_KEY=AIza...\n"
        "Colab: panel de Secrets (ícono 🔑) con Notebook access activado."
    )

# Si ves 429 RESOURCE_EXHAUSTED: crea una API key en un proyecto nuevo en AI Studio,
# o cambia MODEL a "gemini-3.1-flash-lite-preview" (misma sintaxis, más cuota en free tier).
MODEL = "gemini-3.1-flash-lite-preview"
EMBED_MODEL = "gemini-embedding-2"

GOOGLE_API_KEY = load_api_key()
client = genai.Client(api_key=GOOGLE_API_KEY)
print(f"SDK version: {genai.__version__}")
print("Cliente inicializado.")

API key cargada desde .env
SDK version: 1.74.0
Cliente inicializado.


---
## 01 · Primer `generate_content()` — anatomía del response

La API es simple: le mandas `contents` (lo que el usuario dice), recibes un `GenerateContentResponse`.  
Antes de usar el output hay que entender qué tiene ese objeto.

In [3]:
response = client.models.generate_content(
    model=MODEL,
    contents="¿Qué es una transferencia SWIFT y para qué sirve?"
)

# El atajo más común
print(response.text)

Una **transferencia SWIFT** es el método estándar y más utilizado a nivel mundial para enviar dinero de forma segura entre bancos situados en diferentes países.

Aquí te explico los puntos clave para entenderla:

### 1. ¿Qué significa SWIFT?
El término proviene de las siglas en inglés **Society for Worldwide Interbank Financial Telecommunication** (Sociedad para las Telecomunicaciones Financieras Interbancarias Mundiales).

Es importante aclarar que **SWIFT no es un banco**, sino una red de mensajería internacional. Esta red permite que los bancos de todo el mundo se comuniquen entre sí de forma segura para dar instrucciones de pago. Es como un "sistema de mensajería cifrada" para el movimiento de dinero.

### 2. ¿Cómo funciona?
Cuando realizas una transferencia internacional, no envías físicamente el dinero de un banco a otro, sino que se siguen estos pasos:
1.  **Instrucción:** Tu banco envía un mensaje cifrado (a través de la red SWIFT) al banco del destinatario con los detalles de 

In [4]:
# Anatomía completa del response
print("=== CANDIDATES ===")
print(f"  Número de candidatos: {len(response.candidates)}")
candidate = response.candidates[0]
print(f"  finish_reason     : {candidate.finish_reason}")
print(f"  safety_ratings    : {candidate.safety_ratings}")

print()
print("=== USAGE METADATA ===")
usage = response.usage_metadata
print(f"  prompt_token_count    : {usage.prompt_token_count}")
print(f"  candidates_token_count: {usage.candidates_token_count}")
print(f"  total_token_count     : {usage.total_token_count}")

# Nota pedagógica: los tokens son dinero. 1M tokens input = ~$0.075 con Flash.

=== CANDIDATES ===
  Número de candidatos: 1
  finish_reason     : STOP
  safety_ratings    : None

=== USAGE METADATA ===
  prompt_token_count    : 13
  candidates_token_count: 732
  total_token_count     : 745


---
## 02 · Parámetros del LLM — `temperature`, `top_p`, `max_output_tokens`, `system_instruction`

Todo va dentro de `GenerateContentConfig`. No se pasa como argumento separado.

In [ ]:
# temperature = 0 → determinista, siempre la misma respuesta
# Ideal para extracción de datos, compliance, cualquier tarea donde quieres consistencia

config_deterministic = types.GenerateContentConfig(
    temperature=0.0,
    max_output_tokens=512,
    system_instruction="Eres un analista de riesgo de un gran banco peruano. Responde en español, siempre conciso."
)

prompt = "Dame tres señales de alerta de lavado de activos en transacciones de remesas."

r_low = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config=config_deterministic
)
print("=== temperature=0.0 ===")
print(r_low.text)

=== temperature=0.0 ===
Como analista de riesgo, estas son las tres señales de alerta críticas en remesas:

1. **Estructuración (Smurfing):** Múltiples envíos de montos pequeños realizados por diferentes personas hacia un mismo beneficiario, o viceversa, para evitar los umbrales de reporte obligatorio.
2. **Incoherencia con el perfil:** Transacciones que no guardan relación con la capacidad económica, ocupación o actividad declarada del cliente (ej. montos elevados sin sustento de ingresos).
3. **Patrones inusuales de origen/destino:** Recepción o envío frecuente de fondos desde o hacia jurisdicciones de alto riesgo o paraísos fiscales, sin una justificación comercial o familiar lógica.


In [ ]:
# temperature = 1.0 → más variabilidad, más creatividad
# Útil para generación de contenido, brainstorming, datos sintéticos

config_creative = types.GenerateContentConfig(
    temperature=1.0,
    max_output_tokens=512,
    system_instruction="Eres un analista de riesgo de un gran banco peruano. Responde en español, siempre conciso."
)

r_high = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config=config_creative
)
print("=== temperature=1.0 ===")
print(r_high.text)

# Ejecuta esta celda varias veces. ¿Cambia la respuesta? ¿Por qué importa en producción?

=== temperature=1.0 ===
Como analista de riesgo, estas son tres señales de alerta críticas en remesas:

1. **Estructuración (Smurfing):** Múltiples envíos de montos pequeños realizados por diferentes personas a un mismo beneficiario, o a una misma cuenta, para evitar los umbrales de reporte obligatorio.
2. **Incoherencia con el perfil:** Remesas enviadas o recibidas que no guardan relación con la capacidad económica, actividad laboral o nivel de ingresos declarado por el cliente.
3. **Patrones inusuales de cobro:** Receptor que retira el dinero en efectivo de forma inmediata tras la recepción, o que utiliza múltiples ventanillas en distintas zonas geográficas para fraccionar el cobro.


---
## 03 · Outputs estructurados con `response_schema`

En producción **nunca** confías en parsear texto libre. Usas `response_schema` para forzar JSON válido.

> En el SDK nuevo, `response_schema` acepta una clase Pydantic directamente. No necesitas `Schema()` manual.

In [8]:
# Definir el schema con Pydantic
from typing import Optional, Literal

class TransactionInfo(BaseModel):
    entity: str                    # nombre de persona o empresa
    amount: float                  # monto numérico
    currency: Literal["PEN", "USD", "EUR", "OTHER"]
    category: str                  # tipo de operación
    bank: Optional[str] = None     # entidad bancaria si se menciona
    transaction_type: str          # transferencia, pago, retiro, etc.

print("Schema definido:")
print(TransactionInfo.model_json_schema())

Schema definido:
{'properties': {'entity': {'title': 'Entity', 'type': 'string'}, 'amount': {'title': 'Amount', 'type': 'number'}, 'currency': {'enum': ['PEN', 'USD', 'EUR', 'OTHER'], 'title': 'Currency', 'type': 'string'}, 'category': {'title': 'Category', 'type': 'string'}, 'bank': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'title': 'Bank'}, 'transaction_type': {'title': 'Transaction Type', 'type': 'string'}}, 'required': ['entity', 'amount', 'currency', 'category', 'transaction_type'], 'title': 'TransactionInfo', 'type': 'object'}


In [10]:
# Llamada con schema estructurado
transaction_text = "Transferí S/. 4,500 a Inversiones Andinas SAC en Interbank por pago de servicios de TI."

config_structured = types.GenerateContentConfig(
    temperature=0.0,
    response_mime_type="application/json",
    response_schema=TransactionInfo,
    system_instruction="Extrae información de transacciones financieras. Sé preciso con los montos."
)

response = client.models.generate_content(
    model=MODEL,
    contents=f"Extrae los datos de esta transacción: {transaction_text}",
    config=config_structured
)

# El output ya es JSON válido
result = json.loads(response.text)
print(json.dumps(result, indent=2, ensure_ascii=False))

{
  "entity": "Inversiones Andinas SAC",
  "amount": 4500,
  "currency": "PEN",
  "category": "servicios de TI",
  "bank": "Interbank",
  "transaction_type": "transferencia"
}


---
## 04 · Streaming

Para UX en tiempo real (chatbots, reportes largos). En producción casi siempre se usa streaming.

In [ ]:
# generate_content_stream devuelve un iterador de chunks
print("Respuesta en streaming:\n")

for chunk in client.models.generate_content_stream(
    model=MODEL,
    contents="Explica en 3 oraciones qué es el riesgo de crédito en fintech."
):
    print(chunk.text, end="", flush=True)

print("\n\n[stream completado]")

---
## 05 · EJERCICIO 1 — Extracción de entidades de transacciones financieras

**Objetivo:** usar `response_schema` para extraer datos estructurados de texto libre.  
**Tiempo:** ~25 minutos

### Parte A — Extraer datos de múltiples transacciones

Tenemos 5 transacciones en texto. Tu trabajo es:
1. Definir una función `extract_transaction(text: str) -> dict` que use el schema `TransactionInfo`
2. Aplicarla a las 5 transacciones del dataset
3. Armar una lista de resultados

### Parte B — Efecto de temperature

Llama a la misma función con temperature=0 y temperature=1 sobre la transacción TXN-010 (la sospechosa).  
¿Cambia el output? ¿Cuál es el riesgo en producción de usar T>0 para extracción de datos?

In [ ]:
# Transacciones de prueba
transactions = [
    "Transferí S/. 350 a Juan Pérez en BCP para pago de alquiler en Miraflores.",
    "Recibí USD 1,200 desde Rodrigo Lopez en Bank of America por honorarios de consultoría.",
    "Pagué S/. 89.90 en Saga Falabella San Isidro con tarjeta BBVA, compra de ropa.",
    "Envié S/. 12,500 a Inversiones Andinas SAC RUC 20601234567 por adelanto de contrato TI, desde Interbank.",
    "Tres transferencias internacionales en 48 horas: USD 15,000 a Panamá, USD 12,000 a Islas Caimán, USD 18,000 a Panamá. Total: USD 45,000.",
]

In [ ]:
# TODO: Parte A
# 1. Define la función extract_transaction(text: str) -> dict
#    - Usa config_structured de la celda anterior (o crea uno nuevo con T=0)
#    - Retorna el resultado como dict (json.loads del response.text)
# 2. Itera sobre `transactions` y acumula resultados en una lista `results`
# 3. Imprime cada resultado con json.dumps(..., indent=2)

def extract_transaction(text: str) -> dict:
    # TODO: implementar
    pass

results = []
for txn in transactions:
    # TODO: llamar extract_transaction y agregar a results
    pass

print(f"Procesadas: {len(results)} transacciones")

In [ ]:
# TODO: Parte B — temperatura y variabilidad
# Llama a extract_transaction con T=0 y T=1 sobre la TXN-010 (últimas 3 transferencias)
# Ejecuta 3 veces con T=1 y observa si el output cambia
# Reflexión: ¿qué implicaciones tiene esto para un sistema de compliance en producción?

suspicious_txn = transactions[-1]

# TODO: implementar comparación

In [ ]:
# SOLUCIÓN (ejecutar solo después de intentarlo)

def extract_transaction_solution(text: str, temperature: float = 0.0) -> dict:
    config = types.GenerateContentConfig(
        temperature=temperature,
        response_mime_type="application/json",
        response_schema=TransactionInfo,
        system_instruction="Extrae datos de transacciones financieras. Sé preciso con montos y entidades."
    )
    response = client.models.generate_content(
        model=MODEL,
        contents=f"Extrae los datos de esta transacción: {text}",
        config=config
    )
    return json.loads(response.text)

print("=== Todas las transacciones ===")
for i, txn in enumerate(transactions):
    result = extract_transaction_solution(txn)
    print(f"\nTXN-{i+1:03d}:")
    print(json.dumps(result, indent=2, ensure_ascii=False))

---
## 06 · Inputs multimodales — imágenes

Gemini acepta imágenes directamente en `contents`. Hay dos formas:  
1. **Inline (PIL/bytes):** para imágenes pequeñas en memoria, sin persistencia
2. **File API:** para imágenes grandes o reutilizables, se suben una vez y se referencian por URI

En el curso usamos ambas. En producción, File API es lo estándar.

In [3]:
# Método 1: PIL inline — carga la imagen directo como objeto PIL
# En Colab: asegúrate de tener el voucher en el path correcto

# Opción A: desde el repo clonado
# image_path = "genai-multimodal-course/data/images/voucher_yape_001.png"

# Opción B: subir manualmente con el panel de archivos de Colab
# image_path = "voucher_yape_001.png"

# Para la demo, creamos un voucher fake mínimo en memoria
from PIL import Image, ImageDraw

def make_demo_voucher():
    img = Image.new("RGB", (400, 300), color=(102, 45, 145))
    draw = ImageDraw.Draw(img)
    draw.text((10, 10), "YAPE - COMPROBANTE", fill="white")
    draw.text((10, 50), "Fecha: 02/05/2024  09:14", fill="white")
    draw.text((10, 80), "Monto: S/ 350.00", fill="white")
    draw.text((10, 110), "De: Luis Quispe", fill="white")
    draw.text((10, 140), "Para: Ana Torres", fill="white")
    draw.text((10, 170), "Concepto: Pago cuota depa", fill="white")
    draw.text((10, 200), "Estado: EXITOSO", fill="white")
    return img

voucher_img = make_demo_voucher()
voucher_img.show()

# Llamada multimodal: imagen + pregunta
response = client.models.generate_content(
    model=MODEL,
    contents=[voucher_img, "¿Qué información financiera contiene este voucher?"]
)
print(response.text)

Este voucher de Yape contiene la siguiente información financiera:

*   **Fecha y hora:** 02/05/2024 a las 09:14.
*   **Monto:** S/ 350.00 (trescientos cincuenta soles peruanos).
*   **Remitente (De):** Luis Quispe.
*   **Destinatario (Para):** Ana Torres.
*   **Concepto:** Pago cuota depa.
*   **Estado de la transacción:** EXITOSO.


In [4]:
# Método 2: File API — sube el archivo una vez, úsalo múltiples veces
import io

# Convertir PIL image a bytes
buffer = io.BytesIO()
voucher_img.save(buffer, format="PNG")
buffer.seek(0)

# Subir a File API de Google
uploaded_file = client.files.upload(
    file=buffer,
    config=types.UploadFileConfig(
        display_name="voucher_yape_demo",
        mime_type="image/png"
    )
)

print(f"Archivo subido: {uploaded_file.name}")
print(f"URI           : {uploaded_file.uri}")
print(f"MIME type     : {uploaded_file.mime_type}")

# Ahora podemos referenciar el archivo por su nombre
response = client.models.generate_content(
    model=MODEL,
    contents=[
        types.Part.from_uri(file_uri=uploaded_file.uri, mime_type="image/png"),
        "Extrae todos los datos del voucher: monto, origen, destino, fecha y estado."
    ]
)
print()
print(response.text)

Archivo subido: files/4ucl6u1yodbi
URI           : https://generativelanguage.googleapis.com/v1beta/files/4ucl6u1yodbi
MIME type     : image/png


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

---
## 07 · Inputs multimodales — PDFs

Gemini maneja PDFs de forma nativa. El PDF se tokeniza directamente, no se extrae texto antes.

In [7]:
# Demo con el MD de la circular como texto (mismo resultado pedagógico que un PDF)
# En Colab: ajusta el path según dónde hayas subido los archivos del curso
# En VSCode local: usa el path relativo desde la raíz del repo

import os

# Detectar el entorno y armar el path
CIRCULAR_PATH = None
candidates = [
    "data/circulares_sbs/circular_B_2244_2024.md",                        # VSCode / local
    "genai-multimodal-course/data/circulares_sbs/circular_B_2244_2024.md", # Colab con repo clonado
    "/content/circular_B_2244_2024.md",                                    # Colab upload manual
]
for p in candidates:
    if os.path.exists(p):
        CIRCULAR_PATH = p
        break

if CIRCULAR_PATH is None:
    # Fallback: texto embebido directamente (sin archivo)
    circular_text = """
Artículo 4 — Umbral de reporte de transferencias internacionales.
Las empresas supervisadas reportarán toda transferencia internacional que supere USD 10,000
en una sola operación, o cuando la suma de transferencias supere USD 30,000 en 30 días.
El plazo de envío es de 15 días hábiles desde el cierre del mes.
"""
    print("[Usando texto embebido — sube el archivo MD para el ejemplo completo]")
else:
    with open(CIRCULAR_PATH, "r") as f:
        circular_text = f.read()
    print(f"Circular cargada desde: {CIRCULAR_PATH}")
    print(f"Longitud: {len(circular_text)} caracteres")


# Llamada a Gemini con el texto de la circular
response = client.models.generate_content(
    model=MODEL,
    contents=f"""
Analiza el siguiente documento normativo y responde la pregunta.

DOCUMENTO:
{circular_text}

PREGUNTA: ¿Cuál es el umbral en USD para reportar transferencias internacionales a la UIF-Perú?
Cita el artículo exacto que lo establece.
""",
    config=types.GenerateContentConfig(
        temperature=0.0,
        system_instruction="Eres un experto en regulación SBS. Responde en español, sé preciso y cita artículos."
    )
)
print()
print(response.text)


[Usando texto embebido — sube el archivo MD para el ejemplo completo]

De acuerdo con el documento normativo proporcionado, los umbrales para el reporte de transferencias internacionales son los siguientes:

1.  **Operación única:** Toda transferencia que supere los **USD 10,000**.
2.  **Operaciones acumuladas:** Cuando la suma de las transferencias supere los **USD 30,000** en un periodo de 30 días.

El artículo que establece estos umbrales es el **Artículo 4 — Umbral de reporte de transferencias internacionales**.


---
## 08 · EJERCICIO 2 — Voucher → JSON y Circular SBS → extracción

**Objetivo:** combinar inputs multimodales con outputs estructurados.  
**Tiempo:** ~25 minutos

### Parte A — Voucher → JSON
Dado el voucher del BBVA (transferencia internacional de USD 15,000), extrae:
- monto y moneda
- banco y beneficiario de destino
- propósito declarado
- estado de la operación

Usa `response_schema` para que el output sea JSON válido.

### Parte B — Circular SBS → artículos clave
Con la Circular B-2244-2024 cargada en texto, extrae una lista de todos los artículos que mencionan umbrales numéricos (montos o plazos). Output: lista de objetos con `article_number`, `threshold_value`, `threshold_unit`, `description`.

In [8]:
# Schema para el voucher internacional
class VoucherInternacional(BaseModel):
    amount: float
    currency: str
    beneficiary_bank: str
    beneficiary_name: str
    purpose: str
    status: str

class ThresholdArticle(BaseModel):
    article_number: str
    threshold_value: str    # ej: "10000" o "30"
    threshold_unit: str     # ej: "USD", "dias", "horas"
    description: str

class ArticleList(BaseModel):
    articles: list[ThresholdArticle]

In [9]:
# TODO: Parte A — Voucher BBVA → JSON
# 1. Crea un voucher PIL con datos del BBVA (USD 15,000 a Banco Santander Uruguay)
#    O usa directamente el texto descriptivo si no tienes la imagen
# 2. Llama a Gemini con VoucherInternacional como response_schema
# 3. Imprime el resultado

voucher_bbva_text = """
BBVA - SWIFT INTERNACIONAL
Fecha: 28/04/2024 11:05:44
Ref. SWIFT: BSAB20240428PE00192
Tipo: Transferencia internacional
Monto: USD 15,000.00
Equivalente PEN: S/ 56,250.00
T/C aplicado: 3.750
Cuenta origen: 0011-XXXXXXXX-00
Banco beneficiario: Banco Santander Uruguay
SWIFT beneficiario: BSURUS33
Beneficiario: Carlos Mendoza EIRL
Proposito: Pago proveedor materia prima
Estado: EN PROCESO
"""

# TODO: implementar la extracción usando VoucherInternacional

In [10]:
# TODO: Parte B — Circular SBS → lista de artículos con umbrales
# 1. Usa el texto de circular_text cargado en la celda anterior
# 2. Usa ArticleList como response_schema
# 3. Imprime los artículos encontrados

# TODO: implementar

In [11]:
# SOLUCIÓN

# Parte A
config_voucher = types.GenerateContentConfig(
    temperature=0.0,
    response_mime_type="application/json",
    response_schema=VoucherInternacional,
)
r_a = client.models.generate_content(
    model=MODEL,
    contents=f"Extrae los datos de este voucher bancario: {voucher_bbva_text}",
    config=config_voucher
)
print("=== Parte A — Voucher BBVA ===")
print(json.dumps(json.loads(r_a.text), indent=2, ensure_ascii=False))

print()

# Parte B
config_articles = types.GenerateContentConfig(
    temperature=0.0,
    response_mime_type="application/json",
    response_schema=ArticleList,
)
r_b = client.models.generate_content(
    model=MODEL,
    contents=f"""
Del siguiente documento normativo, extrae todos los artículos que mencionan umbrales numéricos.
Un umbral es cualquier monto en USD o PEN, o cualquier plazo en días u horas.

DOCUMENTO:
{circular_text}
""",
    config=config_articles
)
print("=== Parte B — Artículos con umbrales ===")
parsed = json.loads(r_b.text)
for art in parsed["articles"]:
    print(f"  {art['article_number']}: {art['threshold_value']} {art['threshold_unit']} — {art['description'][:60]}...")

=== Parte A — Voucher BBVA ===
{
  "amount": 15000.0,
  "currency": "USD",
  "beneficiary_bank": "Banco Santander Uruguay",
  "beneficiary_name": "Carlos Mendoza EIRL",
  "purpose": "Pago proveedor materia prima",
  "status": "EN PROCESO"
}

=== Parte B — Artículos con umbrales ===
  4: 10000 USD — Umbral de reporte para una sola transferencia internacional...
  4: 30000 USD — Umbral de reporte para la suma de transferencias...
  4: 30 días — Plazo para el cálculo de la suma de transferencias...
  4: 15 días hábiles — Plazo de envío del reporte desde el cierre del mes...


---
## 09 · Embeddings con `gemini-embedding-2`

Un embedding es un vector de números que representa el **significado semántico** de un texto.  
Textos similares → vectores cercanos en el espacio.

**Por qué importa:** los embeddings son el puente entre documentos y LLMs. Sin ellos no hay RAG.

**Honestidad técnica:** `gemini-embedding-2` solo acepta texto. No hay multimodal embeddings nativos en el SDK público.  
La solución estándar en producción (y lo que haremos en sesión 2): Gemini describe la imagen → embed la descripción → guardar referencia a la imagen original. Este es un caso de uso real similar a un problema que tuvimos en Revolut.

In [12]:
# Generar un embedding
result = client.models.embed_content(
    model=EMBED_MODEL,
    contents="transferencia internacional sospechosa lavado de activos"
)

embedding = result.embeddings[0].values
print(f"Tipo       : {type(embedding)}")
print(f"Dimensión  : {len(embedding)}")   # 768 dimensiones
print(f"Min / Max  : {min(embedding):.4f} / {max(embedding):.4f}")
print(f"Primeros 8 : {embedding[:8]}")

Tipo       : <class 'list'>
Dimensión  : 3072
Min / Max  : -0.1823 / 0.1316
Primeros 8 : [-0.012379958, -0.017122675, 0.017019596, 0.024843987, 0.029462384, -0.0089378785, 0.018791948, 0.014499121]


In [33]:
# Batch de embeddings — más eficiente que llamar uno por uno
texts = [
    "transferencia internacional sospechosa lavado de activos",
    "pago de alquiler mensual departamento",
    "reporte a la UIF umbral USD 10000",
    "compra en supermercado con tarjeta de débito",
    "operación en efectivo por encima del límite regulatorio"
]

embeddings = np.array([
    client.models.embed_content(
        model=EMBED_MODEL,
        contents=text
    ).embeddings[0].values
    for text in texts
])
print(f"Shape del batch: {embeddings.shape}")  # (5, 3072)

Shape del batch: (5, 3072)


---
## 10 · Similitud coseno en numpy

La similitud coseno mide el ángulo entre dos vectores. Vale entre -1 y 1.  
**1.0** = idénticos semánticamente · **0.0** = sin relación · **-1.0** = opuestos

$$\text{cos\_sim}(A, B) = \frac{A \cdot B}{\|A\| \cdot \|B\|}$$

In [34]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Similitud coseno entre dos vectores 1D."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Demo: qué tan similares son las frases entre sí
query = embeddings[0]  # "transferencia internacional sospechosa..."

print("Query: 'transferencia internacional sospechosa lavado de activos'\n")
print(f"{'Texto':<55} {'Similitud':>10}")
print("-" * 67)
for i, (text, emb) in enumerate(zip(texts, embeddings)):
    sim = cosine_similarity(query, emb)
    print(f"{text[:54]:<55} {sim:>10.4f}")

Query: 'transferencia internacional sospechosa lavado de activos'

Texto                                                    Similitud
-------------------------------------------------------------------
transferencia internacional sospechosa lavado de activ      1.0000
pago de alquiler mensual departamento                       0.5970
reporte a la UIF umbral USD 10000                           0.7406
compra en supermercado con tarjeta de débito                0.5930
operación en efectivo por encima del límite regulatori      0.7381


---
## 11 · CASO PRÁCTICO — Compliance sin RAG

**Objetivo pedagógico:** que sientas el dolor de hacer esto a mano → motivación real para la sesión 2.

### Escenario

> Un cliente de Revolut Perú realizó **3 transferencias internacionales en 48 horas** por un total de **USD 45,000** hacia Panamá e Islas Caimán. ¿Requiere reporte a la UIF según la normativa SBS vigente?

### Flujo (sin pipeline, todo a mano):
1. Cargar fragmentos de la circular SBS como lista de strings
2. Embed cada fragmento con `gemini-embedding-2`
3. Embed la query del analista
4. Calcular similitud coseno manualmente
5. Seleccionar top-3 fragmentos más relevantes
6. Construir el prompt con el contexto recuperado
7. Llamar a Gemini y obtener respuesta con citación de artículos

**Al final reflexiona:** ¿qué pasa si tienes 500 circulares? ¿Y si vienen 100 queries por minuto?

In [35]:
# Paso 1: fragmentos de la circular SBS (divididos por artículo)
# En producción vendría de tu scraper de SPIJ

circular_chunks = [
    """Artículo 3 — Umbral de reporte de operaciones en efectivo.
Las empresas del sistema financiero reportarán a la UIF-Perú toda transacción en efectivo que supere 
USD 10,000 o su equivalente, en una sola operación o en varias operaciones vinculadas que en conjunto 
superen dicho monto en un período de 30 días calendario. El reporte debe enviarse dentro de los 
15 días hábiles siguientes al cierre del mes.""",

    """Artículo 4 — Umbral de reporte de transferencias internacionales.
Las empresas supervisadas reportarán toda transferencia internacional que supere USD 10,000 en una 
sola operación, o cuando la suma de transferencias realizadas por el mismo titular hacia el mismo 
destino supere USD 30,000 en un período de 30 días calendario. Plazo de envío: 15 días hábiles 
desde el cierre del mes.""",

    """Artículo 5 — Monitoreo de transacciones y alertas automáticas.
Los sistemas de monitoreo deben generar alertas cuando un cliente realice más de 5 transferencias 
internacionales en un período de 48 horas, independientemente del monto individual. Las alertas 
deben ser atendidas por el Oficial de Cumplimiento dentro de los 3 días hábiles siguientes.""",

    """Artículo 6 — Debida diligencia reforzada (DDR).
Se aplicará DDR en operaciones con jurisdicciones incluidas en la lista del GAFI como de alto riesgo. 
Panamá e Islas Caimán están sujetas a monitoreo reforzado. La DDR implica visita al domicilio, 
declaración jurada de origen de fondos y actualización del expediente cada 6 meses.""",

    """Artículo 2 — Definición de operación sospechosa.
Operación sospechosa: aquella respecto de la cual, luego de efectuado el análisis, no existe 
justificación razonable, o que por cualquier motivo genere indicios de vinculación con actividades 
de lavado de activos o financiamiento del terrorismo.""",

    """Artículo 7 — Obligaciones del Oficial de Cumplimiento.
El Oficial de Cumplimiento debe reportar a la UIF-Perú las operaciones sospechosas dentro de las 
24 horas de concluido el análisis, sin informar al cliente. Debe mantener registro de alertas por 
un mínimo de 10 años.""",

    """Artículo 8 — Régimen de sanciones.
El incumplimiento será sancionado con multas entre 5 y 200 UIT. Infracciones graves: omisión de 
reporte de operaciones sobre el umbral. Infracciones muy graves: encubrimiento activo de operaciones 
de lavado de activos.""",

    """Artículo 1 — Objeto y ámbito de aplicación.
La Circular establece requisitos mínimos para la detección, análisis y reporte de operaciones 
sospechosas de lavado de activos y financiamiento del terrorismo. Es de aplicación obligatoria para 
bancos, financieras, cajas municipales, cajas rurales, cooperativas supervisadas y empresas de 
servicios de pago.""",

    """Artículo 5b — Detección de fraccionamiento (structuring).
Los sistemas de monitoreo deben detectar patrones de fraccionamiento de operaciones destinados a 
eludir los umbrales de reporte. También deben identificar transacciones que superen el perfil 
histórico del cliente en más de 3 desviaciones estándar en un período de 90 días.""",

    """Circular SBS B-2251-2024 — Artículo 6 — Reporte de dinero electrónico.
Para billeteras digitales, se reportan como inusuales: cuentas básicas que acumulen más de S/ 1,800 
en movimientos diarios, patrones de microtransacciones (más de 20 operaciones bajo S/ 10 en 2 horas), 
y transferencias circulares que regresen al origen en menos de 24 horas."""
]

print(f"Total de fragmentos: {len(circular_chunks)}")
print(f"Fragmento 0 (primeras 100 chars): {circular_chunks[0][:100]}...")

Total de fragmentos: 10
Fragmento 0 (primeras 100 chars): Artículo 3 — Umbral de reporte de operaciones en efectivo.
Las empresas del sistema financiero repor...


In [36]:
# Paso 2 y 3: embed los fragmentos + la query
query = "Un cliente realizó 3 transferencias internacionales en 48 horas por un total de USD 45,000 hacia Panamá e Islas Caimán. ¿Requiere reporte a la UIF?"

# Embed los fragmentos (un batch)
chunk_embeddings = np.array([
    client.models.embed_content(model=EMBED_MODEL, contents=chunk).embeddings[0].values
    for chunk in circular_chunks
])


# Embed la query
query_result = client.models.embed_content(
    model=EMBED_MODEL,
    contents=query
)
query_embedding = np.array(query_result.embeddings[0].values)

print(f"Chunks embeddings shape: {chunk_embeddings.shape}")
print(f"Query embedding shape  : {query_embedding.shape}")

Chunks embeddings shape: (10, 3072)
Query embedding shape  : (3072,)


In [37]:
# Paso 4 y 5: similitud coseno manual → top-3
similarities = np.array([
    cosine_similarity(query_embedding, chunk_emb)
    for chunk_emb in chunk_embeddings
])

# Ranking
ranked_indices = np.argsort(similarities)[::-1]  # de mayor a menor

print(f"Query: {query[:80]}...\n")
print(f"{'Rank':<5} {'Sim':>7}   {'Fragmento (primeros 80 chars)':<80}")
print("-" * 100)
for rank, idx in enumerate(ranked_indices):
    print(f"{rank+1:<5} {similarities[idx]:>7.4f}   {circular_chunks[idx][:80].replace(chr(10), ' ')}...")

# Seleccionar top-3
top_3_indices = ranked_indices[:3]
top_3_chunks = [circular_chunks[i] for i in top_3_indices]

Query: Un cliente realizó 3 transferencias internacionales en 48 horas por un total de ...

Rank      Sim   Fragmento (primeros 80 chars)                                                   
----------------------------------------------------------------------------------------------------
1      0.8130   Artículo 4 — Umbral de reporte de transferencias internacionales. Las empresas s...
2      0.7719   Artículo 3 — Umbral de reporte de operaciones en efectivo. Las empresas del sist...
3      0.7713   Artículo 5 — Monitoreo de transacciones y alertas automáticas. Los sistemas de m...
4      0.7420   Circular SBS B-2251-2024 — Artículo 6 — Reporte de dinero electrónico. Para bill...
5      0.7274   Artículo 7 — Obligaciones del Oficial de Cumplimiento. El Oficial de Cumplimient...
6      0.7194   Artículo 6 — Debida diligencia reforzada (DDR). Se aplicará DDR en operaciones c...
7      0.7189   Artículo 5b — Detección de fraccionamiento (structuring). Los sistemas de monito...
8      0.6

In [38]:
# Pasos 6 y 7: construir el prompt de augmentation + llamar a Gemini
context = "\n\n---\n\n".join(top_3_chunks)

augmented_prompt = f"""
Eres un analista de compliance del sistema financiero peruano.
Basándote ÚNICAMENTE en la normativa SBS proporcionada a continuación, 
responde la pregunta del analista. Cita el artículo exacto.
Si la normativa no es suficiente para responder, dilo explícitamente.

=== NORMATIVA SBS (fragmentos recuperados) ===
{context}

=== PREGUNTA DEL ANALISTA ===
{query}

=== RESPUESTA ===
"""

response = client.models.generate_content(
    model=MODEL,
    contents=augmented_prompt,
    config=types.GenerateContentConfig(
        temperature=0.0,
        max_output_tokens=512
    )
)

print("=== RESPUESTA DE GEMINI (con contexto normativo) ===")
print(response.text)

=== RESPUESTA DE GEMINI (con contexto normativo) ===
Sí, la operación requiere reporte a la UIF.

De acuerdo con el **Artículo 4 — Umbral de reporte de transferencias internacionales**, las empresas supervisadas deben reportar toda transferencia internacional cuando la suma de transferencias realizadas por el mismo titular hacia el mismo destino supere USD 30,000 en un período de 30 días calendario.

Dado que el cliente realizó transferencias por un total de USD 45,000 (monto que supera el umbral de USD 30,000), la operación debe ser reportada dentro de los 15 días hábiles siguientes al cierre del mes, según lo estipulado en el mismo artículo.

Cabe precisar que, aunque el **Artículo 5** establece alertas automáticas por cantidad de transferencias (más de 5 en 48 horas), en este caso el reporte es obligatorio debido a que se ha superado el umbral monetario acumulado establecido en el **Artículo 4**.


In [39]:
# Reflexión final: el costo de no tener pipeline

print("=" * 60)
print("REFLEXIÓN: ¿Por qué esto no escala?")
print("=" * 60)
print()
print(f"Documentos procesados hoy    : {len(circular_chunks)}")
print(f"Documentos reales en SPIJ    : ~15,000 artículos")
print()
print("Problemas con el enfoque actual:")
print("  1. Re-embeddamos TODO en cada consulta")
print("     → 15,000 * 1 llamada API = lento y caro")
print()
print("  2. No tenemos metadata: ¿de qué circular viene cada fragmento?")
print("     → No podemos filtrar por año, tipo de norma, etc.")
print()
print("  3. No hay persistencia: si reiniciamos Colab, perdemos todo")
print()
print("  4. np.argsort sobre 15,000 vectores → O(n) en cada query")
print("     → Un vector store hace esto en O(log n) con índices HNSW")
print()
print("Mañana: ChromaDB resuelve los 4 problemas en ~20 líneas.")

REFLEXIÓN: ¿Por qué esto no escala?

Documentos procesados hoy    : 10
Documentos reales en SPIJ    : ~15,000 artículos

Problemas con el enfoque actual:
  1. Re-embeddamos TODO en cada consulta
     → 15,000 * 1 llamada API = lento y caro

  2. No tenemos metadata: ¿de qué circular viene cada fragmento?
     → No podemos filtrar por año, tipo de norma, etc.

  3. No hay persistencia: si reiniciamos Colab, perdemos todo

  4. np.argsort sobre 15,000 vectores → O(n) en cada query
     → Un vector store hace esto en O(log n) con índices HNSW

Mañana: ChromaDB resuelve los 4 problemas en ~20 líneas.


---

## Fin de Sesión 1

**Qué construiste hoy:**
- Llamadas a Gemini con control de parámetros
- Extracción de datos estructurados de texto e imágenes
- Sistema de compliance básico con embeddings y similitud coseno

**Sesión 2 (mañana):** ChromaDB + RAG completo + multimodal + FastAPI demo.

---
*Dudas o feedback: rlopezvera98@gmail.com